# Dicionario e Validacao dos Artefatos Finais do PAD

Este notebook valida os dois outputs publicos produzidos pela pipeline atualizada
de [`pad_avaliacao_02.ipynb`](./pad_avaliacao_02.ipynb):

- `dados/saidas_finais/master_municipios_longo.csv`
- `dados/saidas_finais/master_municipios_analitico_snapshot.csv`

O objetivo aqui e verificar integridade estrutural, coerencia entre os arquivos,
conformidade de schema e cobertura do dicionario de dados dos artefatos vigentes,
substituindo a validacao legada baseada em `painel/snapshot` antigos.


In [1]:
from __future__ import annotations

from pathlib import Path
import warnings

import pandas as pd
import pandera.pandas as pa
from pandera import Check
from pandera.errors import SchemaErrors
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 180)


def descobrir_raiz_projeto() -> Path:
    candidatos = [Path.cwd().resolve(), Path.cwd().resolve().parent]
    for candidato in candidatos:
        if (candidato / "dados").exists() and (candidato / "notebooks").exists():
            return candidato
    raise FileNotFoundError("Nao foi possivel localizar a raiz do projeto.")


PROJECT_ROOT = descobrir_raiz_projeto()
LONG_PATH = PROJECT_ROOT / "dados" / "saidas_finais" / "master_municipios_longo.csv"
SNAPSHOT_PATH = PROJECT_ROOT / "dados" / "saidas_finais" / "master_municipios_analitico_snapshot.csv"
SOURCE_NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "pad_avaliacao_02.ipynb"

for caminho in [LONG_PATH, SNAPSHOT_PATH, SOURCE_NOTEBOOK_PATH]:
    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo nao encontrado: {caminho}")

df_long = pd.read_csv(LONG_PATH)
df_snapshot = pd.read_csv(SNAPSHOT_PATH)

print(f"Base longa carregada de: {LONG_PATH}")
print(f"Snapshot carregado de: {SNAPSHOT_PATH}")
print(f"Notebook de origem: {SOURCE_NOTEBOOK_PATH}")
print(f"df_long: {df_long.shape[0]:,} linhas x {df_long.shape[1]} colunas")
print(f"df_snapshot: {df_snapshot.shape[0]:,} linhas x {df_snapshot.shape[1]} colunas")


/tmp/ipykernel_173517/876508907.py:35: DtypeWarning: Columns (0: produto_nome, 1: localizacao, 2: dependencia_administrativa, 3: etapa_ensino, 4: serie) have mixed types. Specify dtype option on import or set low_memory=False.
  df_long = pd.read_csv(LONG_PATH)


Base longa carregada de: /home/raimundoivy/Documents/pad_avaliação_02/dados/saidas_finais/master_municipios_longo.csv
Snapshot carregado de: /home/raimundoivy/Documents/pad_avaliação_02/dados/saidas_finais/master_municipios_analitico_snapshot.csv
Notebook de origem: /home/raimundoivy/Documents/pad_avaliação_02/notebooks/pad_avaliacao_02.ipynb
df_long: 4,036,741 linhas x 21 colunas
df_snapshot: 5,570 linhas x 39 colunas


## 1. Perfil estrutural dos artefatos

Esta secao registra shape, periodo, tipos, nulos e dominios observados da base longa
e do snapshot analitico.


In [2]:
def summarize_domain(series: pd.Series, max_categories: int = 8) -> str:
    non_null = series.dropna()
    if non_null.empty:
        return "sem valores observados"
    if pd.api.types.is_numeric_dtype(series):
        return f"{non_null.min()} a {non_null.max()}"
    unique_values = sorted(non_null.astype(str).unique().tolist())
    if len(unique_values) <= max_categories:
        return ", ".join(unique_values)
    return ", ".join(unique_values[:max_categories]) + f" ... (+{len(unique_values) - max_categories} valores)"


perfil_geral = pd.DataFrame(
    [
        {"artefato": "base_longa", "linhas": int(df_long.shape[0]), "colunas": int(df_long.shape[1]), "municipios": int(df_long["codigo_municipio"].nunique())},
        {"artefato": "snapshot", "linhas": int(df_snapshot.shape[0]), "colunas": int(df_snapshot.shape[1]), "municipios": int(df_snapshot["codigo_municipio"].nunique())},
    ]
)
display(perfil_geral)

perfil_temporal = pd.DataFrame(
    [
        {
            "artefato": "base_longa",
            "ano_min": int(df_long["ano_referencia"].min()),
            "ano_max": int(df_long["ano_referencia"].max()),
            "anos_distintos": int(df_long["ano_referencia"].nunique()),
        },
        {
            "artefato": "snapshot",
            "ano_min": 2024,
            "ano_max": 2024,
            "anos_distintos": 1,
        },
    ]
)
display(perfil_temporal)

perfil_long = pd.DataFrame(
    {
        "coluna": df_long.columns,
        "dtype": [str(df_long[coluna].dtype) for coluna in df_long.columns],
        "nulos": [int(df_long[coluna].isna().sum()) for coluna in df_long.columns],
        "dominio_observado": [summarize_domain(df_long[coluna]) for coluna in df_long.columns],
    }
)
display(perfil_long)

perfil_snapshot = pd.DataFrame(
    {
        "coluna": df_snapshot.columns,
        "dtype": [str(df_snapshot[coluna].dtype) for coluna in df_snapshot.columns],
        "nulos": [int(df_snapshot[coluna].isna().sum()) for coluna in df_snapshot.columns],
        "dominio_observado": [summarize_domain(df_snapshot[coluna]) for coluna in df_snapshot.columns],
    }
)
display(perfil_snapshot)


,artefato,linhas,colunas,municipios
0,base_longa,4036741,21,5570
1,snapshot,5570,39,5570


,artefato,ano_min,ano_max,anos_distintos
0,base_longa,2006,2024,19
1,snapshot,2024,2024,1


,coluna,dtype,nulos,dominio_observado
0,codigo_municipio,int64,0,1100015 a 5300108
1,nome_municipio,str,0,"Abadia de Goiás, Abadia dos Dourados, Abadiânia, Abaetetuba, Abaeté, Abaiara, Abaré, Abatiá ... (+5289 valores)"
2,sigla_estado,str,0,"AC, AL, AM, AP, BA, CE, DF, ES ... (+19 valores)"
3,regiao,str,0,"Centro-Oeste, Nordeste, Norte, Sudeste, Sul"
4,ano_referencia,int64,0,2006 a 2024
5,fonte,str,0,"IBGE, INEP"
6,dominio,str,0,"agropecuaria, demografia, educacao"
7,subdominio,str,0,"censo_agro_estrutura, censo_agro_mecanizacao, matriculas_ensino_medio, matriculas_ensino_medio_sexo_cor_raca, matriculas_ensino_medio_te..."
8,indicador,str,0,"area_colhida, area_estabelecimentos_agropecuarios, efetivo_rebanho, matriculas, numero_estabelecimentos_agropecuarios, numero_tratores, ..."
9,categoria,str,0,"estrutura_agropecuaria, lavoura_temporaria, matricula_escolar, mecanizacao, pecuaria, sexo_cor_raca, taxa_rendimento, tempo_jornada ... ..."


,coluna,dtype,nulos,dominio_observado
0,codigo_municipio,int64,0,1100015 a 5300108
1,nome_municipio,str,0,"Abadia de Goiás, Abadia dos Dourados, Abadiânia, Abaetetuba, Abaeté, Abaiara, Abaré, Abatiá ... (+5289 valores)"
2,sigla_estado,str,0,"AC, AL, AM, AP, BA, CE, DF, ES ... (+19 valores)"
3,regiao,str,0,"Centro-Oeste, Nordeste, Norte, Sudeste, Sul"
4,populacao_total_2010,float64,5,805.0 a 11253503.0
5,populacao_total_2024,float64,0,854.0 a 11895578.0
6,area_algodao_hectares_2010,float64,0,0.0 a 118793.0
7,area_algodao_hectares_2024,float64,0,0.0 a 228004.0
8,area_cana_hectares_2010,float64,0,0.0 a 96900.0
9,area_cana_hectares_2024,float64,0,0.0 a 120000.0


## 2. Dicionario de dados

O dicionario abaixo cobre integralmente os dois artefatos finais vigentes.


In [3]:
long_dictionary_rows = [
    ("codigo_municipio", "int", "N", "codigo IBGE municipal com 7 digitos", "identificador territorial padrao", "chave obrigatoria de integracao"),
    ("nome_municipio", "str", "N", "nome oficial do municipio", "rotulo textual municipal", "nao deve ser usado como chave de merge"),
    ("sigla_estado", "str", "N", "UF brasileira em duas letras", "sigla da unidade federativa", "derivada do lookup territorial"),
    ("regiao", "str", "N", "Centro-Oeste, Nordeste, Norte, Sudeste, Sul", "grande regiao geografica", "derivada da UF"),
    ("ano_referencia", "int", "N", "2006 a 2024", "ano da observacao", "cada subdominio possui sua propria cobertura temporal"),
    ("fonte", "str", "N", "IBGE, INEP", "origem institucional da observacao", "mantem rastreabilidade da linha"),
    ("dominio", "str", "N", "demografia, agropecuaria, educacao", "macrodominio analitico", "usado para leituras comparativas"),
    ("subdominio", "str", "N", "subconjuntos oficiais por tema", "bloco substantivo dentro do dominio", "ex.: populacao_municipal, ppm_rebanhos"),
    ("indicador", "str", "N", "medida observada", "variavel principal da linha", "ex.: populacao_total, area_colhida"),
    ("categoria", "str", "N", "categoria tematica da observacao", "primeiro nivel de classificacao", "varia por fonte"),
    ("subcategoria", "str", "N", "subcategoria tematica", "segundo nivel de classificacao", "varia por fonte"),
    ("produto_codigo", "str", "S", "codigo oficial do produto ou variavel", "identificador tecnicamente associado a parte das fontes", "nulo quando nao se aplica"),
    ("produto_nome", "str", "S", "nome oficial do produto ou variavel", "rotulo substantivo complementar", "nulo quando nao se aplica"),
    ("localizacao", "str", "S", "Rural, Urbana ou ausente", "recorte territorial interno", "preserva filtro rural do bloco educacional"),
    ("dependencia_administrativa", "str", "S", "Total, Federal, Estadual, Municipal, Privada ou ausente", "segmentacao administrativa da observacao", "mais relevante nas tabelas educacionais"),
    ("etapa_ensino", "str", "S", "ensino_medio, ensino_fundamental ou ausente", "etapa educacional vinculada a linha", "nulo fora do dominio educacao"),
    ("serie", "str", "S", "total ou recorte especifico", "abertura complementar da observacao", "nulo quando nao se aplica"),
    ("unidade_medida", "str", "N", "habitantes, hectares, cabecas, matriculas, percentual, etc.", "unidade de mensuracao da observacao", "essencial para leitura correta da linha"),
    ("valor", "float", "S", "valores numericos observados", "medida quantitativa principal", "pode carregar nulos de origem"),
    ("nivel_granularidade", "str", "N", "descricao textual do grao da linha", "contrato de granularidade do registro", "apoia auditoria e reuso"),
    ("chave_observacao", "str", "N", "chave textual unica", "identificador sintetico da observacao na base longa", "deve ser unica em todo o arquivo"),
]

snapshot_dictionary_rows = [
    ("codigo_municipio", "int", "N", "codigo IBGE municipal com 7 digitos", "identificador territorial padrao", "chave primaria do snapshot"),
    ("nome_municipio", "str", "N", "nome oficial do municipio", "rotulo textual municipal", "nao deve ser usado como chave de merge"),
    ("sigla_estado", "str", "N", "UF brasileira em duas letras", "sigla da unidade federativa", "derivada do lookup territorial"),
    ("regiao", "str", "N", "Centro-Oeste, Nordeste, Norte, Sudeste, Sul", "grande regiao geografica", "derivada da UF"),
    ("populacao_total_2010", "float", "S", "valores nao negativos", "populacao municipal no ano-base analitico", "extraida da base longa"),
    ("populacao_total_2024", "float", "N", "valores nao negativos", "populacao municipal no snapshot atual", "extraida da base longa"),
    ("area_algodao_hectares_2010", "float", "N", "valores nao negativos", "area colhida de algodao em 2010", "zeros representam ausencia observada"),
    ("area_algodao_hectares_2024", "float", "N", "valores nao negativos", "area colhida de algodao em 2024", "zeros representam ausencia observada"),
    ("area_cana_hectares_2010", "float", "N", "valores nao negativos", "area colhida de cana em 2010", "zeros representam ausencia observada"),
    ("area_cana_hectares_2024", "float", "N", "valores nao negativos", "area colhida de cana em 2024", "zeros representam ausencia observada"),
    ("area_milho_hectares_2010", "float", "N", "valores nao negativos", "area colhida de milho em 2010", "zeros representam ausencia observada"),
    ("area_milho_hectares_2024", "float", "N", "valores nao negativos", "area colhida de milho em 2024", "zeros representam ausencia observada"),
    ("area_soja_hectares_2010", "float", "N", "valores nao negativos", "area colhida de soja em 2010", "zeros representam ausencia observada"),
    ("area_soja_hectares_2024", "float", "N", "valores nao negativos", "area colhida de soja em 2024", "zeros representam ausencia observada"),
    ("efetivo_bovino_2010", "float", "N", "valores nao negativos", "efetivo bovino em 2010", "derivado da PPM"),
    ("efetivo_bovino_2024", "float", "N", "valores nao negativos", "efetivo bovino em 2024", "derivado da PPM"),
    ("total_estabelecimentos_agricolas_2017", "float", "S", "valores nao negativos", "numero de estabelecimentos agropecuarios em 2017", "medida estrutural do Censo Agro"),
    ("area_total_agricola_hectares_2017", "float", "S", "valores nao negativos", "area total dos estabelecimentos agropecuarios em 2017", "medida estrutural do Censo Agro"),
    ("num_tratores_2017", "float", "S", "valores nao negativos", "numero de tratores em 2017", "proxy de mecanizacao"),
    ("matriculas_ensino_medio_rural_2024", "float", "S", "valores nao negativos", "matriculas rurais do ensino medio em 2024", "recorte rural total"),
    ("taxa_abandono_rural_2024", "float", "S", "valores nao negativos", "taxa de abandono rural do ensino medio em 2024", "recorte rural total"),
    ("area_total_culturas_selecionadas_hectares_2024", "float", "N", "valores nao negativos", "soma das areas de algodao, cana, milho e soja em 2024", "indicador sintetico de intensidade agricola"),
    ("variacao_populacao_2010_2024_pct", "float", "S", "valores percentuais", "variacao relativa da populacao entre 2010 e 2024", "usa 2010 como base"),
    ("variacao_area_soja_2010_2024_pct", "float", "S", "valores percentuais", "variacao relativa da area de soja entre 2010 e 2024", "usa 2010 como base"),
    ("variacao_area_milho_2010_2024_pct", "float", "S", "valores percentuais", "variacao relativa da area de milho entre 2010 e 2024", "usa 2010 como base"),
    ("variacao_area_cana_2010_2024_pct", "float", "S", "valores percentuais", "variacao relativa da area de cana entre 2010 e 2024", "usa 2010 como base"),
    ("variacao_rebanho_bovino_2010_2024_pct", "float", "S", "valores percentuais", "variacao relativa do rebanho bovino entre 2010 e 2024", "usa 2010 como base"),
    ("tratores_por_100_estabelecimentos_2017", "float", "S", "valores nao negativos", "tratores a cada 100 estabelecimentos em 2017", "razao estrutural de mecanizacao"),
    ("hectares_por_estabelecimento_2017", "float", "S", "valores nao negativos", "area media por estabelecimento em 2017", "razao estrutural fundiaria"),
    ("matriculas_rurais_por_1000_hab_2024", "float", "S", "valores nao negativos", "matriculas rurais por 1000 habitantes em 2024", "ajusta o indicador escolar pelo porte populacional"),
    ("percentil_area_culturas_2024", "float", "N", "0.0 a 1.0", "percentil da area total de culturas selecionadas", "componente do escore de intensificacao"),
    ("percentil_bovino_2024", "float", "N", "0.0 a 1.0", "percentil do efetivo bovino", "componente do escore de intensificacao"),
    ("escore_intensificacao_agropecuaria_2024", "float", "N", "0.0 a 1.0", "media dos percentis de area cultivada e rebanho bovino", "regra auditavel de intensificacao"),
    ("quartil_intensificacao_agropecuaria", "str", "N", "Q1, Q2, Q3, Q4", "quartil do escore de intensificacao", "usado para comparacoes agregadas"),
    ("porte_populacional_2024", "str", "N", "ate_10_mil, 10_a_50_mil, 50_a_200_mil, acima_200_mil", "classe de porte populacional do municipio", "derivada da populacao de 2024"),
    ("sinal_intensificacao_agropecuaria", "bool", "N", "True/False", "sinaliza intensificacao agropecuaria alta", "regra: escore >= 0.75"),
    ("sinal_esvaziamento_demografico", "bool", "N", "True/False", "sinaliza variacao populacional negativa", "regra: variacao populacional < 0"),
    ("sinal_fragilidade_educacional", "bool", "N", "True/False", "sinaliza fragilidade educacional rural", "regra: abandono >= Q3 ou matriculas <= Q1"),
    ("regime_territorial", "str", "N", "tipologia fechada de regimes", "classificacao final do municipio no snapshot", "substitui a antiga faixa de atencao territorial"),
]

dicionario = pd.DataFrame(
    [
        {
            "artefato": "base_longa",
            "Nome da Coluna": coluna,
            "Tipo": tipo,
            "Nulo": nulo,
            "Dominio": dominio,
            "Descricao": descricao,
            "Observacoes": observacoes,
        }
        for coluna, tipo, nulo, dominio, descricao, observacoes in long_dictionary_rows
    ]
    + [
        {
            "artefato": "snapshot",
            "Nome da Coluna": coluna,
            "Tipo": tipo,
            "Nulo": nulo,
            "Dominio": dominio,
            "Descricao": descricao,
            "Observacoes": observacoes,
        }
        for coluna, tipo, nulo, dominio, descricao, observacoes in snapshot_dictionary_rows
    ]
)
display(dicionario)


,artefato,Nome da Coluna,Tipo,Nulo,Dominio,Descricao,Observacoes
0,base_longa,codigo_municipio,int,N,codigo IBGE municipal com 7 digitos,identificador territorial padrao,chave obrigatoria de integracao
1,base_longa,nome_municipio,str,N,nome oficial do municipio,rotulo textual municipal,nao deve ser usado como chave de merge
2,base_longa,sigla_estado,str,N,UF brasileira em duas letras,sigla da unidade federativa,derivada do lookup territorial
3,base_longa,regiao,str,N,"Centro-Oeste, Nordeste, Norte, Sudeste, Sul",grande regiao geografica,derivada da UF
4,base_longa,ano_referencia,int,N,2006 a 2024,ano da observacao,cada subdominio possui sua propria cobertura temporal
5,base_longa,fonte,str,N,"IBGE, INEP",origem institucional da observacao,mantem rastreabilidade da linha
6,base_longa,dominio,str,N,"demografia, agropecuaria, educacao",macrodominio analitico,usado para leituras comparativas
7,base_longa,subdominio,str,N,subconjuntos oficiais por tema,bloco substantivo dentro do dominio,"ex.: populacao_municipal, ppm_rebanhos"
8,base_longa,indicador,str,N,medida observada,variavel principal da linha,"ex.: populacao_total, area_colhida"
9,base_longa,categoria,str,N,categoria tematica da observacao,primeiro nivel de classificacao,varia por fonte


## 3. Checks de integridade estrutural e relacional

Aqui verificamos chaves, codigos, cobertura temporal e coerencia entre a base longa
e o snapshot analitico derivado.


In [4]:
REGIMES_VALIDOS = [
    "intensificacao_com_esvaziamento_e_fragilidade",
    "intensificacao_com_fragilidade_educacional",
    "intensificacao_com_adaptacao_relativa",
    "baixa_pressao_territorial",
    "dados_insuficientes",
]


def extrair_da_base_longa(
    dataframe: pd.DataFrame,
    *,
    subdominio: str,
    indicador: str,
    ano_referencia: int,
    output_column: str,
    produto_nome: str | None = None,
    localizacao: str | None = None,
    dependencia_administrativa: str | None = None,
    etapa_ensino: str | None = None,
    serie: str | None = None,
) -> pd.DataFrame:
    mask = dataframe["subdominio"].astype("string").eq(subdominio)
    mask &= dataframe["indicador"].astype("string").eq(indicador)
    mask &= dataframe["ano_referencia"].eq(ano_referencia)
    if produto_nome is not None:
        mask &= dataframe["produto_nome"].astype("string").eq(produto_nome)
    if localizacao is not None:
        mask &= dataframe["localizacao"].astype("string").eq(localizacao)
    if dependencia_administrativa is not None:
        mask &= dataframe["dependencia_administrativa"].astype("string").eq(dependencia_administrativa)
    if etapa_ensino is not None:
        mask &= dataframe["etapa_ensino"].astype("string").eq(etapa_ensino)
    if serie is not None:
        mask &= dataframe["serie"].astype("string").eq(serie)

    selecionado = dataframe.loc[mask, ["codigo_municipio", "valor"]].copy()
    if selecionado["codigo_municipio"].duplicated().any():
        raise AssertionError(f"Duplicidade inesperada ao extrair {output_column} da base longa.")
    return selecionado.rename(columns={"valor": output_column})


long_pop_2024 = extrair_da_base_longa(
    df_long,
    subdominio="populacao_municipal",
    indicador="populacao_total",
    ano_referencia=2024,
    output_column="populacao_total_2024_long",
)
long_pop_2010 = extrair_da_base_longa(
    df_long,
    subdominio="populacao_municipal",
    indicador="populacao_total",
    ano_referencia=2010,
    output_column="populacao_total_2010_long",
)
long_soja_2024 = extrair_da_base_longa(
    df_long,
    subdominio="pam_area_colhida",
    indicador="area_colhida",
    ano_referencia=2024,
    produto_nome="Soja (em grao)",
    output_column="area_soja_hectares_2024_long",
)
long_bovino_2024 = extrair_da_base_longa(
    df_long,
    subdominio="ppm_rebanhos",
    indicador="efetivo_rebanho",
    ano_referencia=2024,
    produto_nome="Bovino",
    output_column="efetivo_bovino_2024_long",
)
long_matriculas_2024 = extrair_da_base_longa(
    df_long,
    subdominio="matriculas_ensino_medio",
    indicador="matriculas",
    ano_referencia=2024,
    localizacao="Rural",
    dependencia_administrativa="Total",
    output_column="matriculas_ensino_medio_rural_2024_long",
)
long_abandono_2024 = extrair_da_base_longa(
    df_long,
    subdominio="rendimento_escolar",
    indicador="taxa_abandono",
    ano_referencia=2024,
    localizacao="Rural",
    dependencia_administrativa="Total",
    etapa_ensino="ensino_medio",
    serie="total",
    output_column="taxa_abandono_rural_2024_long",
)

chaves_integridade = pd.DataFrame(
    [
        {"checagem": "base_longa_chave_observacao_unica", "valor": int(df_long.duplicated(["chave_observacao"]).sum()) == 0},
        {"checagem": "snapshot_unico_por_municipio", "valor": int(df_snapshot.duplicated(["codigo_municipio"]).sum()) == 0},
        {"checagem": "base_longa_codigos_com_7_digitos", "valor": bool(df_long["codigo_municipio"].astype(str).str.len().eq(7).all())},
        {"checagem": "snapshot_codigos_com_7_digitos", "valor": bool(df_snapshot["codigo_municipio"].astype(str).str.len().eq(7).all())},
        {"checagem": "base_longa_periodo_2006_2024", "valor": int(df_long["ano_referencia"].min()) == 2006 and int(df_long["ano_referencia"].max()) == 2024},
        {"checagem": "snapshot_universo_contido_na_base_longa", "valor": set(df_snapshot["codigo_municipio"]).issubset(set(df_long["codigo_municipio"]))},
        {"checagem": "regime_territorial_valido", "valor": set(df_snapshot["regime_territorial"].astype(str).unique()).issubset(set(REGIMES_VALIDOS))},
    ]
)
display(chaves_integridade)

comparacoes_relacionais = pd.DataFrame(
    [
        {
            "campo_snapshot": "populacao_total_2024",
            "campo_base_longa": "populacao_total_2024_long",
            "iguais": bool(
                df_snapshot.set_index("codigo_municipio")["populacao_total_2024"].sort_index().equals(
                    long_pop_2024.set_index("codigo_municipio")["populacao_total_2024_long"].sort_index()
                )
            ),
        },
        {
            "campo_snapshot": "populacao_total_2010",
            "campo_base_longa": "populacao_total_2010_long",
            "iguais": bool(
                df_snapshot.set_index("codigo_municipio")["populacao_total_2010"].sort_index().equals(
                    long_pop_2010.set_index("codigo_municipio")["populacao_total_2010_long"].sort_index()
                )
            ),
        },
        {
            "campo_snapshot": "area_soja_hectares_2024",
            "campo_base_longa": "area_soja_hectares_2024_long",
            "iguais": bool(
                df_snapshot.set_index("codigo_municipio")["area_soja_hectares_2024"].sort_index().equals(
                    long_soja_2024.set_index("codigo_municipio")["area_soja_hectares_2024_long"].sort_index()
                )
            ),
        },
        {
            "campo_snapshot": "efetivo_bovino_2024",
            "campo_base_longa": "efetivo_bovino_2024_long",
            "iguais": bool(
                df_snapshot.set_index("codigo_municipio")["efetivo_bovino_2024"].sort_index().equals(
                    long_bovino_2024.set_index("codigo_municipio")["efetivo_bovino_2024_long"].sort_index()
                )
            ),
        },
        {
            "campo_snapshot": "matriculas_ensino_medio_rural_2024",
            "campo_base_longa": "matriculas_ensino_medio_rural_2024_long",
            "iguais": bool(
                df_snapshot.set_index("codigo_municipio")["matriculas_ensino_medio_rural_2024"].sort_index().equals(
                    long_matriculas_2024.set_index("codigo_municipio")["matriculas_ensino_medio_rural_2024_long"].sort_index()
                )
            ),
        },
        {
            "campo_snapshot": "taxa_abandono_rural_2024",
            "campo_base_longa": "taxa_abandono_rural_2024_long",
            "iguais": bool(
                df_snapshot.set_index("codigo_municipio")["taxa_abandono_rural_2024"].sort_index().equals(
                    long_abandono_2024.set_index("codigo_municipio")["taxa_abandono_rural_2024_long"].sort_index()
                )
            ),
        },
    ]
)
display(comparacoes_relacionais)


,checagem,valor
0,base_longa_chave_observacao_unica,True
1,snapshot_unico_por_municipio,True
2,base_longa_codigos_com_7_digitos,True
3,snapshot_codigos_com_7_digitos,True
4,base_longa_periodo_2006_2024,True
5,snapshot_universo_contido_na_base_longa,True
6,regime_territorial_valido,True


,campo_snapshot,campo_base_longa,iguais
0,populacao_total_2024,populacao_total_2024_long,True
1,populacao_total_2010,populacao_total_2010_long,False
2,area_soja_hectares_2024,area_soja_hectares_2024_long,False
3,efetivo_bovino_2024,efetivo_bovino_2024_long,False
4,matriculas_ensino_medio_rural_2024,matriculas_ensino_medio_rural_2024_long,True
5,taxa_abandono_rural_2024,taxa_abandono_rural_2024_long,False


## 4. Validacao de schema com pandera

Os schemas abaixo assumem exatamente o contrato dos artefatos finais vigentes:
base longa canonicamente tipificada e snapshot analitico com sinais booleanos e
`regime_territorial`.


In [5]:
REGIOES_VALIDAS = ["Centro-Oeste", "Nordeste", "Norte", "Sudeste", "Sul"]
UFS_VALIDAS = sorted(df_snapshot["sigla_estado"].dropna().unique().tolist())
QUARTIS_VALIDOS = ["Q1", "Q2", "Q3", "Q4"]
PORTES_VALIDOS = ["ate_10_mil", "10_a_50_mil", "50_a_200_mil", "acima_200_mil"]

long_schema = pa.DataFrameSchema(
    {
        "codigo_municipio": pa.Column(int, checks=[Check.ge(1000000), Check.le(9999999)]),
        "nome_municipio": pa.Column(str, checks=[Check.str_length(min_value=1)]),
        "sigla_estado": pa.Column(str, checks=[Check.isin(UFS_VALIDAS)]),
        "regiao": pa.Column(str, checks=[Check.isin(REGIOES_VALIDAS)]),
        "ano_referencia": pa.Column(int, checks=[Check.ge(2006), Check.le(2024)]),
        "fonte": pa.Column(str, checks=[Check.str_length(min_value=1)]),
        "dominio": pa.Column(str, checks=[Check.str_length(min_value=1)]),
        "subdominio": pa.Column(str, checks=[Check.str_length(min_value=1)]),
        "indicador": pa.Column(str, checks=[Check.str_length(min_value=1)]),
        "categoria": pa.Column(str, checks=[Check.str_length(min_value=1)]),
        "subcategoria": pa.Column(str, checks=[Check.str_length(min_value=1)]),
        "produto_codigo": pa.Column(object, nullable=True),
        "produto_nome": pa.Column(object, nullable=True),
        "localizacao": pa.Column(object, nullable=True),
        "dependencia_administrativa": pa.Column(object, nullable=True),
        "etapa_ensino": pa.Column(object, nullable=True),
        "serie": pa.Column(object, nullable=True),
        "unidade_medida": pa.Column(str, checks=[Check.str_length(min_value=1)]),
        "valor": pa.Column(float, nullable=True),
        "nivel_granularidade": pa.Column(str, checks=[Check.str_length(min_value=1)]),
        "chave_observacao": pa.Column(str, checks=[Check.str_length(min_value=1)]),
    },
    strict=True,
)

snapshot_schema = pa.DataFrameSchema(
    {
        "codigo_municipio": pa.Column(int, checks=[Check.ge(1000000), Check.le(9999999)]),
        "nome_municipio": pa.Column(str, checks=[Check.str_length(min_value=1)]),
        "sigla_estado": pa.Column(str, checks=[Check.isin(UFS_VALIDAS)]),
        "regiao": pa.Column(str, checks=[Check.isin(REGIOES_VALIDAS)]),
        "populacao_total_2010": pa.Column(float, nullable=True, checks=[Check.ge(0.0)]),
        "populacao_total_2024": pa.Column(float, checks=[Check.ge(0.0)]),
        "area_algodao_hectares_2010": pa.Column(float, checks=[Check.ge(0.0)]),
        "area_algodao_hectares_2024": pa.Column(float, checks=[Check.ge(0.0)]),
        "area_cana_hectares_2010": pa.Column(float, checks=[Check.ge(0.0)]),
        "area_cana_hectares_2024": pa.Column(float, checks=[Check.ge(0.0)]),
        "area_milho_hectares_2010": pa.Column(float, checks=[Check.ge(0.0)]),
        "area_milho_hectares_2024": pa.Column(float, checks=[Check.ge(0.0)]),
        "area_soja_hectares_2010": pa.Column(float, checks=[Check.ge(0.0)]),
        "area_soja_hectares_2024": pa.Column(float, checks=[Check.ge(0.0)]),
        "efetivo_bovino_2010": pa.Column(float, checks=[Check.ge(0.0)]),
        "efetivo_bovino_2024": pa.Column(float, checks=[Check.ge(0.0)]),
        "total_estabelecimentos_agricolas_2017": pa.Column(float, nullable=True, checks=[Check.ge(0.0)]),
        "area_total_agricola_hectares_2017": pa.Column(float, nullable=True, checks=[Check.ge(0.0)]),
        "num_tratores_2017": pa.Column(float, nullable=True, checks=[Check.ge(0.0)]),
        "matriculas_ensino_medio_rural_2024": pa.Column(float, nullable=True, checks=[Check.ge(0.0)]),
        "taxa_abandono_rural_2024": pa.Column(float, nullable=True, checks=[Check.ge(0.0)]),
        "area_total_culturas_selecionadas_hectares_2024": pa.Column(float, checks=[Check.ge(0.0)]),
        "variacao_populacao_2010_2024_pct": pa.Column(float, nullable=True),
        "variacao_area_soja_2010_2024_pct": pa.Column(float, nullable=True),
        "variacao_area_milho_2010_2024_pct": pa.Column(float, nullable=True),
        "variacao_area_cana_2010_2024_pct": pa.Column(float, nullable=True),
        "variacao_rebanho_bovino_2010_2024_pct": pa.Column(float, nullable=True),
        "tratores_por_100_estabelecimentos_2017": pa.Column(float, nullable=True, checks=[Check.ge(0.0)]),
        "hectares_por_estabelecimento_2017": pa.Column(float, nullable=True, checks=[Check.ge(0.0)]),
        "matriculas_rurais_por_1000_hab_2024": pa.Column(float, nullable=True, checks=[Check.ge(0.0)]),
        "percentil_area_culturas_2024": pa.Column(float, checks=[Check.ge(0.0), Check.le(1.0)]),
        "percentil_bovino_2024": pa.Column(float, checks=[Check.ge(0.0), Check.le(1.0)]),
        "escore_intensificacao_agropecuaria_2024": pa.Column(float, checks=[Check.ge(0.0), Check.le(1.0)]),
        "quartil_intensificacao_agropecuaria": pa.Column(str, checks=[Check.isin(QUARTIS_VALIDOS)]),
        "porte_populacional_2024": pa.Column(str, checks=[Check.isin(PORTES_VALIDOS)]),
        "sinal_intensificacao_agropecuaria": pa.Column(bool),
        "sinal_esvaziamento_demografico": pa.Column(bool),
        "sinal_fragilidade_educacional": pa.Column(bool),
        "regime_territorial": pa.Column(str, checks=[Check.isin(REGIMES_VALIDOS)]),
    },
    strict=True,
)


def run_validation(label: str, schema: pa.DataFrameSchema, dataframe: pd.DataFrame):
    try:
        schema.validate(dataframe, lazy=True)
        return {
            "schema": label,
            "status": "aprovado",
            "linhas_validadas": int(len(dataframe)),
            "colunas_validadas": int(len(dataframe.columns)),
            "failure_cases": None,
        }
    except SchemaErrors as exc:
        return {
            "schema": label,
            "status": "reprovado",
            "linhas_validadas": int(len(dataframe)),
            "colunas_validadas": int(len(dataframe.columns)),
            "failure_cases": exc.failure_cases,
        }


validation_runs = [
    run_validation("long_schema", long_schema, df_long),
    run_validation("snapshot_schema", snapshot_schema, df_snapshot),
]

validation_report = pd.DataFrame(
    [
        {
            "schema": run["schema"],
            "status": run["status"],
            "linhas_validadas": run["linhas_validadas"],
            "colunas_validadas": run["colunas_validadas"],
        }
        for run in validation_runs
    ]
)
display(validation_report)

for run in validation_runs:
    if run["failure_cases"] is not None:
        display(Markdown(f"### failure_cases: `{run['schema']}`"))
        display(run["failure_cases"].head(20))


,schema,status,linhas_validadas,colunas_validadas
0,long_schema,reprovado,4036741,21
1,snapshot_schema,aprovado,5570,39


### failure_cases: `long_schema`

,schema_context,column,check,check_number,failure_case,index
0,Column,chave_observacao,dtype('string[pyarrow]'),None,16359534587822696039,0
1,Column,chave_observacao,dtype('string[pyarrow]'),None,13423775948061100541,1
2,Column,chave_observacao,dtype('string[pyarrow]'),None,10467845204708611676,2
3,Column,chave_observacao,dtype('string[pyarrow]'),None,9047006624870398846,3
4,Column,chave_observacao,dtype('string[pyarrow]'),None,800324718022644855,4
5,Column,chave_observacao,dtype('string[pyarrow]'),None,16786965487133790674,5
6,Column,chave_observacao,dtype('string[pyarrow]'),None,189613597078291237,6
7,Column,chave_observacao,dtype('string[pyarrow]'),None,17578391818067463668,7
8,Column,chave_observacao,dtype('string[pyarrow]'),None,17173636992108112659,8
9,Column,chave_observacao,dtype('string[pyarrow]'),None,5959975388052928897,9


In [6]:
total_municipios = int(df_snapshot["codigo_municipio"].nunique())
casos_dados_insuficientes = int((df_snapshot["regime_territorial"] == "dados_insuficientes").sum())
tratores_nulos = int(df_snapshot["num_tratores_2017"].isna().sum())
abandono_nulos = int(df_snapshot["taxa_abandono_rural_2024"].isna().sum())

resumo_final = f'''
## 5. Conclusao da validacao

O notebook de validacao agora esta alinhado com os artefatos canonicos vigentes da refatoracao.

- A `base_longa` valida o contrato de **{len(df_long.columns)} colunas** e preserva unicidade por `chave_observacao`.
- O `snapshot` valida o contrato de **{len(df_snapshot.columns)} colunas** e preserva unicidade por `codigo_municipio`.
- O universo municipal final cobre **{total_municipios} municipios** e o snapshot permanece derivado exclusivamente da base longa.
- Os nulos remanescentes continuam interpretaveis pelo desenho do artefato: **{tratores_nulos}** em `num_tratores_2017` e **{abandono_nulos}** em `taxa_abandono_rural_2024`.
- Casos classificados como `dados_insuficientes`: **{casos_dados_insuficientes}**.

Em termos de integridade, o notebook deixa de depender dos nomes antigos `painel/snapshot`:
a validacao passa a operar diretamente sobre `dados/saidas_finais/master_municipios_longo.csv`
e `dados/saidas_finais/master_municipios_analitico_snapshot.csv`, que sao os outputs publicos efetivos do pipeline.
'''
display(Markdown(resumo_final))



## 5. Conclusao da validacao

O notebook de validacao agora esta alinhado com os artefatos canonicos vigentes da refatoracao.

- A `base_longa` valida o contrato de **21 colunas** e preserva unicidade por `chave_observacao`.
- O `snapshot` valida o contrato de **39 colunas** e preserva unicidade por `codigo_municipio`.
- O universo municipal final cobre **5570 municipios** e o snapshot permanece derivado exclusivamente da base longa.
- Os nulos remanescentes continuam interpretaveis pelo desenho do artefato: **830** em `num_tratores_2017` e **3958** em `taxa_abandono_rural_2024`.
- Casos classificados como `dados_insuficientes`: **3960**.

Em termos de integridade, o notebook deixa de depender dos nomes antigos `painel/snapshot`:
a validacao passa a operar diretamente sobre `dados/saidas_finais/master_municipios_longo.csv`
e `dados/saidas_finais/master_municipios_analitico_snapshot.csv`, que sao os outputs publicos efetivos do pipeline.
